# Gold: clusters de conteúdo (`governor_clusters_reels` / `governor_clusters_posts`)

ADR [0022](../../docs/adr/0022-notebooks-de-diagnostico-medallion-separados-da-narrativa-do-tcc.md)
(issue [#128](https://github.com/Vini0606/Tecnicas-de-Ciencia-de-Dados-em-dados-do-Instagram/issues/128)).
Diagnóstico dos clusters de conteúdo (saída de `src/modeling/clustering.py`), granularidade de **um
post/reel por linha**. `cluster_label == -1` é ruído do DBSCAN, não um grupo real -- o vocabulário do
produto (`CONTEXT.md`) chama isso de "casos atípicos / virais" na interface final, nunca "-1" cru;
aqui mantemos o rótulo técnico, mas vale lembrar o significado ao interpretar.

Desde a issue [#152](https://github.com/Vini0606/Tecnicas-de-Ciencia-de-Dados-em-dados-do-Instagram/issues/152),
o que antes era uma única tabela `governor_clusters` discriminada por `content_type` virou duas
tabelas Gold separadas, uma por formato -- `posts_clean`/`reels_clean` (Silver) se sobrepõem (um Reel
também é capturado pelo post-scraper genérico no grid do perfil), então a tabela combinada duplicava
o mesmo post real sob os dois `content_type`. Este notebook carrega as duas tabelas separadamente e
combina o resultado só para as seções de distribuição/covariáveis abaixo (leitura, não escrita) --
os dois clusterings continuam sendo análises legitimamente diferentes sobre o mesmo espaço de
conteúdo, não um dado duplicado.

Notebook estritamente leitura via `DeltaRepository`, mesmo princípio da ADR
[0003](../../docs/adr/0003-desacoplar-modelagem-do-notebook-via-scripts-cli-com-checkpoint.md).


In [1]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from deltalake import DeltaTable
from dotenv import load_dotenv

from config import settings
from src.repositories.delta_repository import DeltaRepository
from src.analysis.medallion_diagnostics import (
    completeness_summary,
    count_duplicate_rows,
    with_governor_metadata,
)

load_dotenv()
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

repo = DeltaRepository(gold_dir=settings.GOLD_DIR, silver_dir=settings.SILVER_DIR)

## 1. Carga + schema

In [ ]:
df_clusters_reels = repo.load_clusters_reels()
df_clusters_posts = repo.load_clusters_posts()
# Combinação só para os diagnósticos de distribuição/covariáveis abaixo (leitura,
# não escrita) -- desde a issue #152 as duas granularidades vivem em tabelas Gold
# separadas, sem sobreposição de post real entre si (cada uma vem de um único
# pipeline de clusterização, sobre uma única fonte Silver).
df_clusters = pd.concat([df_clusters_reels, df_clusters_posts], ignore_index=True)
df_clusters.dtypes


## 2. Completude

In [ ]:
for nome, df in [("governor_clusters_reels", df_clusters_reels), ("governor_clusters_posts", df_clusters_posts)]:
    print(f"--- {nome} ---")
    completude = completeness_summary(df)
    # issue #152: cada tabela agora vem de um único pipeline de clusterização
    # sobre uma única fonte Silver -- duplicata por id_reel DENTRO de uma
    # mesma tabela não deveria mais acontecer (era o sintoma da tabela
    # combinada antiga). Confirmar 0 aqui é o teste de regressão da issue.
    print(f"linhas duplicadas (por id_reel): {count_duplicate_rows(df, subset=['id_reel'])}")
    display(completude[completude['n_nulos'] > 0])


## 3. Distribuição: tamanho de cada cluster, por tipo de conteúdo

Feed (posts estáticos) e reel são clusterizados separadamente (`src/modeling/clustering.py`) e,
desde a issue #152, vivem em tabelas Gold separadas -- comparar contagem por `content_type` na
visão combinada (`df_clusters`, seção 1) evita ler um cluster de vídeo como se fosse do mesmo
espaço que um cluster de post estático.


In [ ]:
pd.crosstab(df_clusters['cluster_label'], df_clusters['content_type'])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df_clusters['cluster_score'], kde=False, ax=ax, bins=20)
ax.set_title('Distribuição de cluster_score (score de ajuste do algoritmo)')
plt.tight_layout()
plt.show()
print(df_clusters.groupby('cluster_algo')['cluster_score'].describe())

## 4. Evolução temporal

Sem tabela `_history` irmã para `governor_clusters` hoje -- seção do esqueleto padrão pulada de
propósito.

## 5. Relação com covariáveis (partido/UF)

`governor_clusters_reels`/`governor_clusters_posts` só têm `ownerUsername`, não `inputUrl` --
diferente das outras tabelas Gold deste conjunto de notebooks. Ponte via `profiles_clean` (Silver,
tem os dois) antes de juntar com `governors_metadata`. Usa a visão combinada (`df_clusters`) das
duas tabelas, já que a ponte por `ownerUsername` vale igual para reel e feed.


In [ ]:
mapa_username_inputurl = DeltaTable(str(settings.SILVER_PROFILES)).to_pandas()[['username', 'inputUrl']]
df_clusters_com_url = df_clusters.merge(
    mapa_username_inputurl, left_on='ownerUsername', right_on='username', how='left'
)
taxa_match = df_clusters_com_url['inputUrl'].notna().mean() * 100
print(f'{taxa_match:.1f}% das linhas de governor_clusters encontraram inputUrl em profiles_clean')

df_clusters_metadado = with_governor_metadata(df_clusters_com_url, repo.load_governors_metadata())
pd.crosstab(df_clusters_metadado['partido'], df_clusters_metadado['cluster_label'], normalize='index').mul(100).round(1)

## 6. Outliers

Posts marcados `cluster_label == -1` ("casos atípicos / virais", no vocabulário do produto) e
linhas sem `inputUrl` encontrado no passo anterior (governador que não bateu com `profiles_clean` --
merece checar se é um `ownerUsername` desatualizado, não um bug silencioso).

In [ ]:
atipicos = df_clusters[df_clusters['cluster_label'] == -1]
print(f'{len(atipicos)} posts em cluster_label == -1')
display(atipicos[['ownerUsername', 'content_type', 'cluster_algo', 'cluster_score']])

sem_match = df_clusters_com_url[df_clusters_com_url['inputUrl'].isna()]
print(f'{len(sem_match)} linhas sem inputUrl correspondente em profiles_clean')
sem_match['ownerUsername'].value_counts()

## Nota de interpretação

`cluster_score` vem do algoritmo inteiro (DBSCAN/Agglomerative), não por post individual -- valores
praticamente idênticos entre linhas do mesmo `content_type`/execução são esperados, não um sinal de
pouca variância real nos dados. O que varia de fato é `cluster_label`.